In [11]:
# [Cell 1: Fetching 3-Year Hourly Archive & Building 6,570 Sub-Daily Training Rows]
import pandas as pd
import numpy as np
import requests

print("📂 Loading 3-year hourly weather dataset...")
weather_df = pd.read_csv("pune_historical_weather_2023_2025.csv")
weather_df['date'] = pd.to_datetime(weather_df['date'])
if weather_df['date'].dt.tz is not None:
    weather_df['date'] = weather_df['date'].dt.tz_localize(None)

# 1. Fetch 3-Year HOURLY Air Quality Archive from Open-Meteo (2023-01-01 to 2025-12-31)
print("🌐 Fetching 3-year HOURLY historical air quality archive from Open-Meteo...")
aq_archive_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": 18.5204,
    "longitude": 73.8567,
    "start_date": "2023-01-01",
    "end_date": "2025-12-31",
    "hourly": "pm2_5,pm10,nitrogen_dioxide,us_aqi",
    "timezone": "Asia/Kolkata"
}

resp = requests.get(aq_archive_url, params=params, timeout=15)
resp.raise_for_status()
pollution_df = pd.DataFrame(resp.json()["hourly"])
pollution_df['date'] = pd.to_datetime(pollution_df['time'])
pollution_df = pollution_df.drop(columns=['time'])

# 2. Convert Wind Direction into Directional Vectors (U, V)
rad = np.deg2rad(weather_df['wind_direction'])
weather_df['wind_u'] = -weather_df['wind_speed'] * np.sin(rad)
weather_df['wind_v'] = -weather_df['wind_speed'] * np.cos(rad)

# 3. Resample Weather Data into 4-Hour Intervals
weather_df['time_4h'] = weather_df['date'].dt.floor('4h')
weather_4h = weather_df.groupby('time_4h').agg(
    temp=('temperature', 'mean'),
    humidity=('humidity', 'mean'),
    rain=('rain', 'sum'),
    wind_u=('wind_u', 'mean'),
    wind_v=('wind_v', 'mean'),
    wind_speed=('wind_speed', 'max')
).reset_index()

# 4. Resample HOURLY Pollution Data into 4-Hour Intervals
pollution_df['time_4h'] = pollution_df['date'].dt.floor('4h')
pollution_4h = pollution_df.groupby('time_4h').agg(
    us_aqi=('us_aqi', 'mean'),
    pm2_5=('pm2_5', 'mean'),
    pm10=('pm10', 'mean'),
    nitrogen_dioxide=('nitrogen_dioxide', 'mean')
).reset_index()

# 5. Merge Weather + Pollution on 4-Hour Timestamps
df = pd.merge(weather_4h, pollution_4h, on='time_4h', how='inner').sort_values('time_4h').reset_index(drop=True)
df = df.dropna().reset_index(drop=True)

print(f"🎉 SUCCESS! Created true 4-hour sub-daily training dataset with {len(df)} rows!")
display(df.head(6))

📂 Loading 3-year hourly weather dataset...
🌐 Fetching 3-year HOURLY historical air quality archive from Open-Meteo...
🎉 SUCCESS! Created true 4-hour sub-daily training dataset with 6575 rows!


,time_4h,temp,humidity,rain,wind_u,wind_v,wind_speed,us_aqi,pm2_5,pm10,nitrogen_dioxide
0,2023-01-01 00:00:00,16.6375,78.570564,0.0,1.584,-0.234000,5.351785,152.25,73.850,106.800,57.150
1,2023-01-01 04:00:00,26.6225,38.983812,0.0,0.756,-0.900000,5.991594,152.00,89.250,128.900,53.850
2,2023-01-01 08:00:00,28.8525,28.625054,0.0,6.282,-1.979998,8.699793,151.75,54.900,80.025,32.500
3,2023-01-01 12:00:00,21.3025,44.393380,0.0,6.930,-2.304001,10.703569,151.00,29.125,42.675,2.725
4,2023-01-01 16:00:00,18.3975,56.590281,0.0,1.800,-3.204000,9.021574,150.75,37.300,53.750,18.250
5,2023-01-01 20:00:00,16.7450,64.965120,0.0,-0.198,-2.988000,7.771331,151.25,64.375,92.275,63.425


In [12]:
# [Cell 2: Engineering Sub-Daily Features & 18 Multi-Output Target Columns]
print("⚙️ Engineering sub-daily features and 18-step direct targets...")

# 1. Rain Binary Flag
df['is_raining'] = (df['rain'] > 0).astype(int)

# 2. Cyclical Month Encoding
df['month'] = df['time_4h'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# 3. Cyclical Hour-of-Day Encoding (0, 4, 8, 12, 16, 20)
df['hour'] = df['time_4h'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# 4. Sub-Daily Lags (Lag 1 = 4h ago, Lag 6 = 24h ago)
for col in ['us_aqi', 'pm2_5']:
    df[f'{col}_lag_1step'] = df[col].shift(1)
    df[f'{col}_lag_6step'] = df[col].shift(6)

# 5. DIRECT MULTI-OUTPUT TARGETS: Generate 18 future step targets (3 days x 6 slots/day)
TARGET_COLS = []
for step in range(1, 19): # Steps 1 to 18
    col_name = f'Target_AQI_Step_{step}'
    df[col_name] = df['us_aqi'].shift(-step)
    TARGET_COLS.append(col_name)

# Drop NA rows resulting from shifts
df_model = df.dropna().reset_index(drop=True)

print(f"✅ Created 18 direct target columns: {TARGET_COLS[:3]} ... {TARGET_COLS[-1]}")
print(f"Valid multi-output training rows: {len(df_model)}")

⚙️ Engineering sub-daily features and 18-step direct targets...
✅ Created 18 direct target columns: ['Target_AQI_Step_1', 'Target_AQI_Step_2', 'Target_AQI_Step_3'] ... Target_AQI_Step_18
Valid multi-output training rows: 6551


In [13]:
# [Cell 3: Feature Selection & Chronological Split]
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Target Matrix Y (18 columns)
Y = df_model[TARGET_COLS]

# Feature Matrix X
BINARY_CYCLICAL_FEATURES = ['is_raining', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos']
CONTINUOUS_FEATURES = [
    'temp', 'humidity', 'rain', 'wind_u', 'wind_v', 'wind_speed',
    'us_aqi_lag_1step', 'us_aqi_lag_6step',
    'pm2_5_lag_1step', 'pm2_5_lag_6step'
]

FEATURE_COLS = CONTINUOUS_FEATURES + BINARY_CYCLICAL_FEATURES
X = df_model[FEATURE_COLS]

# Chronological Train-Test Split (80% Train, 20% Test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
Y_train, Y_test = Y.iloc[:split_idx], Y.iloc[split_idx:]

print(f"Train set: {len(X_train)} samples | Test set: {len(X_test)} samples")
print(f"Target shape: {Y_train.shape} (18 output steps per sample)")

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), CONTINUOUS_FEATURES),
        ('passthrough', 'passthrough', BINARY_CYCLICAL_FEATURES)
    ]
)

Train set: 5240 samples | Test set: 1311 samples
Target shape: (5240, 18) (18 output steps per sample)


In [14]:
# [Cell 4: Benchmarking 4 Multi-Output Regression Models]
from sklearn.model_selection import TimeSeriesSplit
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

models = {
    "Ridge Regression": MultiOutputRegressor(Ridge(alpha=1.0)),
    "Support Vector Regressor (SVR)": MultiOutputRegressor(SVR(kernel='rbf', C=100, gamma='scale')),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42), # RF handles multi-output natively
    "XGBoost": MultiOutputRegressor(XGBRegressor(n_estimators=150, learning_rate=0.05, random_state=42))
}

tscv = TimeSeriesSplit(n_splits=5)
results = []

print("⏳ Benchmarking Multi-Output Models across 18 target time steps...")

for name, model_inst in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model_inst)
    ])
    
    maes, rmses, r2s = [], [], []
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        Y_tr, Y_te = Y.iloc[train_idx], Y.iloc[test_idx]
        
        pipe.fit(X_tr, Y_tr)
        preds = pipe.predict(X_te)
        
        maes.append(mean_absolute_error(Y_te, preds))
        rmses.append(root_mean_squared_error(Y_te, preds))
        r2s.append(r2_score(Y_te, preds))
        
    results.append({
        "Model": name,
        "Avg MAE (18 Steps)": round(np.mean(maes), 2),
        "Avg RMSE (18 Steps)": round(np.mean(rmses), 2),
        "Avg R2 Score": round(np.mean(r2s), 3)
    })

results_df = pd.DataFrame(results).sort_values("Avg MAE (18 Steps)")
print("\n=== MULTI-OUTPUT MODEL BENCHMARK RESULTS ===")
display(results_df)

⏳ Benchmarking Multi-Output Models across 18 target time steps...

=== MULTI-OUTPUT MODEL BENCHMARK RESULTS ===


,Model,Avg MAE (18 Steps),Avg RMSE (18 Steps),Avg R2 Score
0,Ridge Regression,14.56,18.86,0.727
3,XGBoost,14.59,19.30,0.709
2,Random Forest,14.88,19.45,0.709
1,Support Vector Regressor (SVR),15.04,19.97,0.693


In [15]:
# [Cell 5: Exporting Champion Multi-Output Pipeline]
import joblib
import os

best_model_name = results_df.iloc[0]['Model']
print(f"🏆 Champion Multi-Output Model Selected: {best_model_name}")

champion_model_inst = models[best_model_name]

# Final Pipeline trained on full X, Y
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', champion_model_inst)
])

final_pipeline.fit(X, Y)

export_path = os.path.join("aqi_forecast_model.pkl") # or 'data/aqi_forecast_model.pkl'
joblib.dump(final_pipeline, export_path)

print(f"🎉 Success! Champion Direct Multi-Output Pipeline exported to '{export_path}'")

🏆 Champion Multi-Output Model Selected: Ridge Regression
🎉 Success! Champion Direct Multi-Output Pipeline exported to 'aqi_forecast_model.pkl'
